# 🔧 題目 3：音樂串流趨勢分析 — Solution

⚠️ 講師用。學員請用 `pipeline_starter.ipynb`。


## Section 0：環境設定


In [ ]:
import pandas as pd
import sqlite3
import os
import json
print('✅ 套件載入完成')


In [ ]:
OPENAI_API_KEY = ""
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]
print("✅" if OPENAI_API_KEY else "⚠️ fallback")


---
## Section 1：Extract


### Step 1-1：讀取 CSV


In [ ]:
df_raw = pd.read_csv("tracks.csv")
print(f"📊 {len(df_raw)} 筆, {len(df_raw.columns)} 欄")
df_raw.head()


### Step 1-2：檢查


In [ ]:
print(df_raw.dtypes)
print("\n", df_raw.isnull().sum())
print("\n", df_raw.describe())


### Step 1-3：自由探索


In [ ]:
print(df_raw.iloc[:, 0].value_counts().head(10))
print(f"\n唯一值: {df_raw.iloc[:, 0].nunique()}")


### Step 1-4：SQLite


In [ ]:
conn = sqlite3.connect("pipeline.db")
df_raw.to_sql("raw_tracks", conn, if_exists="replace", index=False)
print(f"✅ raw_tracks: {pd.read_sql('SELECT COUNT(*) as n FROM raw_tracks', conn)['n'][0]} 筆")


---
## Section 2：Transform


### Step 2-1：從 raw 讀出


In [ ]:
df = pd.read_sql("SELECT * FROM raw_tracks", conn)
before = len(df)
print(f"讀出 {before} 筆")


### Step 2-2 ~ 2-4：清洗


In [ ]:
df = df.dropna(subset=["Track", "Artist"])
for col in ["Spotify Streams", "YouTube Views", "TikTok Posts", "Shazam Counts"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(",", "")
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
df["All Time Rank"] = pd.to_numeric(df["All Time Rank"], errors="coerce")
df["Release Date"] = pd.to_datetime(df["Release Date"], errors="coerce")
df["release_year"] = df["Release Date"].dt.year


In [ ]:
print(f"清洗前: {before} → 清洗後: {len(df)}")


### 🏁 檢查點


In [ ]:
assert df.isnull().sum()[["Track","Artist"]].sum() == 0, "❌ Track/Artist 有空值"
assert "release_year" in df.columns, "❌ 缺少 release_year"
print("✅ 通過")
print(f"   {len(df)} 筆, {len(df.columns)} 欄")


### Step 2-5：寫入 cleaned


In [ ]:
df.to_sql("cleaned_tracks", conn, if_exists="replace", index=False)
print(f"✅ cleaned_tracks")


---
## Section 3：SQL


### Step 3-1：Spotify 播放量 Top 20


In [ ]:
top_tracks = pd.read_sql("""
SELECT Track, Artist, "Spotify Streams" as streams, "All Time Rank" as rank
FROM cleaned_tracks
ORDER BY "Spotify Streams" DESC
LIMIT 20
""", conn)
top_tracks


### Step 3-2：跨平台比較


In [ ]:
artist_cross = pd.read_sql("""
SELECT Artist,
       SUM("Spotify Streams") as spotify,
       SUM("YouTube Views") as youtube,
       SUM("TikTok Posts") as tiktok
FROM cleaned_tracks
GROUP BY Artist
ORDER BY spotify DESC
LIMIT 15
""", conn)
artist_cross


### Step 3-3：視覺化


In [ ]:
import matplotlib.pyplot as plt
top_tracks.head(10).plot.barh(x=top_tracks.columns[0], y=top_tracks.columns[-1], figsize=(10,5))
plt.tight_layout()
plt.show()


### Step 3-5：存結果


In [ ]:
os.makedirs("processed", exist_ok=True)
top_tracks.to_csv("processed/top_tracks.csv", index=False)
artist_cross.to_csv("processed/artist_cross.csv", index=False)
print("✅ 已存")


---
## Section 4：LLM


In [ ]:
import requests
def llm_analyze(text, api_key=None):
    if api_key: return _llm_api(text, api_key)
    return _llm_fallback(text)
def _llm_api(text, api_key):
    prompt = f"""請分析以下歌曲，回傳 JSON：
{{"genre_guess": "流行/嘻哈/搖滾/電子/R&B/其他", "insight": "一句話歌曲洞察"}}\n文字：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3}, timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
        if content.startswith("```"): content = content.split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(content)
    except: return _llm_fallback(text)
def _llm_fallback(text):
    t = text.lower()
    if any(w in t for w in ["love","heart","baby","kiss"]): cat = "流行"
    elif any(w in t for w in ["money","gang","trap","drip"]): cat = "嘻哈"
    elif any(w in t for w in ["rock","metal","punk","guitar"]): cat = "搖滾"
    elif any(w in t for w in ["beat","bass","drop","remix"]): cat = "電子"
    else: cat = "其他"
    return {"genre_guess": cat, "insight": text[:50] + "..."}
print("✅ LLM Helper")


### Step 4-1：單筆測試


In [ ]:
test = str(df["Track"].iloc[0])
result = llm_analyze(test, OPENAI_API_KEY if OPENAI_API_KEY else None)
print(f"📝 {test[:60]}\n🤖 {result}")


### Step 4-2：批次


In [ ]:
BATCH_SIZE = 50
api_key = OPENAI_API_KEY if OPENAI_API_KEY else None
results = []
for i, row in df.head(BATCH_SIZE).iterrows():
    r = llm_analyze(str(row["Track"]), api_key)
    results.append(r)
    if len(results) % 10 == 0: print(f"  {len(results)}/{BATCH_SIZE}")
print(f"✅ {len(results)} 筆")


### Step 4-3：整理 + 寫入


In [ ]:
df_analyzed = df.head(BATCH_SIZE).copy()
first_keys = list(results[0].keys())
for k in first_keys:
    df_analyzed[k] = [r.get(k, "") for r in results]
df_analyzed.rename(columns={k: "llm_insight" for k in first_keys if "insight" in k}, inplace=True)
df_analyzed.to_sql("analyzed_tracks", conn, if_exists="replace", index=False)
print("📊 三表：")
for t in ["raw_tracks", "cleaned_tracks", "analyzed_tracks"]:
    print(f"  {t}: {pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', conn)['n'][0]}")


---
## Section 5：驗證


In [ ]:
lineage = pd.read_sql("""
SELECT \'raw_tracks\' as layer, COUNT(*) as rows FROM raw_tracks
UNION ALL SELECT \'cleaned_tracks\', COUNT(*) FROM cleaned_tracks
UNION ALL SELECT \'analyzed_tracks\', COUNT(*) FROM analyzed_tracks
""", conn)
print(lineage.to_string(index=False))


---
## Section 6：報告


In [ ]:
top3 = top_tracks.head(3)
report = f"""# 音樂串流趨勢分析報告
## 資料概要
- 分析歌曲：{len(df)} 首
## Spotify Top 3
{{chr(10).join(f'- {{r["Track"]}} by {{r["Artist"]}}' for _, r in top3.iterrows())}}
## 建議
1. 關注跨平台表現差異
2. 追蹤新發行歌曲的竄升速度
## Pipeline
CSV → pandas → SQLite → SQL → LLM → 本報告
"""
os.makedirs("output", exist_ok=True)
with open("output/report.md", "w") as f: f.write(report)
print("✅ report.md")


---
## Section 7：打包


In [ ]:
checks = [("pipeline.db","DB"), ("processed","統計"), ("output/report.md","報告")]
for p,d in checks: print(f"  {'✅' if os.path.exists(p) else '❌'} {d}: {p}")
c = sqlite3.connect("pipeline.db")
for t in ["raw_tracks","cleaned_tracks","analyzed_tracks"]:
    try: print(f"  ✅ {t}: {pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', c)['n'][0]}")
    except: print(f"  ❌ {t}")
c.close()
